<div style="font-family:'Times New Roman',Times,serif;color:#000000">
<div style="font-size:13px;letter-spacing:2px">NATIONAL UNIVERSITY OF SCIENCES AND TECHNOLOGY &nbsp;&middot;&nbsp; SEECS</div>
<div style="font-size:13px;letter-spacing:1px;font-weight:bold">CS-423 &nbsp;&middot;&nbsp; ARTIFICIAL NEURAL NETWORKS &amp; DEEP LEARNING &nbsp;&middot;&nbsp; BSAI</div>
<div style="font-size:28px;font-weight:bold;margin-top:6px">Lab 7 &mdash; Open-Ended Lab: Hyperparameter Ablation Study on Handwritten Digits</div>
<div style="font-size:15px;margin-top:4px">3 hours &nbsp;&middot;&nbsp; CLO 2, CLO 3, CLO 4</div>
</div>

---

<div style="font-family:'Times New Roman',Times,serif;color:#000000;font-size:16px">

**Course Instructor:** Dr. Sadia Shakil &nbsp;|&nbsp; **Lab Engineer:** Ms Areeba Rameen &nbsp;|&nbsp; **Student name / CMS ID:** _______________



**What you will do, in short**

1. Take your Lab 6 MLP and add a few extra options (activation, loss, mini-batches, L2).
2. Measure a **baseline** with 5-fold cross-validation.
3. Change **one setting at a time** (an *ablation study*) and see how the result changes.
4. Combine the best settings, test the final model **once** on unseen data, and explain your choices.

**Rules**

- The network must be **your own NumPy code** built on Labs 3 to 6. No Keras, PyTorch or scikit-learn MLP for the experiments. (`pandas` may be used only to show result tables.)
- **Do not touch `X_test` / `y_test` before Task 6.**
- In the ablation, **change one hyperparameter at a time**; everything else stays at the baseline.
- Always use the same `SEED` and the same splitter `skf`, so that results are comparable.
- Report results as **mean &plusmn; std** over the 5 folds. Every plot needs axis labels, a legend and a title.
- Every claim needs a number or a plot as evidence. Work individually.

**Submission.** Restart the kernel, run everything top to bottom, save, and rename to
`Lab7_<yourCMSID>.ipynb` before uploading to LMS.

<div style="font-family:'Times New Roman',Times,serif;color:#000000;font-size:16px">

## Setup

</div>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import pandas as pd
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split, StratifiedKFold
np.set_printoptions(precision=4, suppress=True)

SEED = 42
digits = load_digits()
X = digits.data                                   # shape (1797, 64), pixel values 0..16
digit = digits.target                             # the real digit 0-9 (used only for looking at errors)
y = (digit % 2).reshape(-1, 1).astype(float)      # 1 = odd, 0 = even

# Lock away 20% as the test set. Do NOT use X_test / y_test before Task 6.
X_dev, X_test, y_dev, y_test, d_dev, d_test = train_test_split(
    X, y, digit, test_size=0.20, stratify=y, random_state=SEED)

def standardise(X_fit, *others):
    """Scale with the mean/std of X_fit ONLY; apply the same scaling to the others."""
    mu, sd = X_fit.mean(axis=0), X_fit.std(axis=0)
    sd[sd == 0] = 1.0                              # pixels that are always 0
    return [(A - mu) / sd for A in (X_fit,) + others]

# Use exactly this splitter for every cross-validation in this lab
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

BASELINE = dict(hidden=16, lr=0.1, epochs=500, activation="sigmoid", loss="bce",
                batch_size=None, l2=0.0, init_scale=0.5, seed=0)

def show_images(X_raw, titles, idx, ncols=5):
    """Show the 8x8 images X_raw[idx] with the given titles (provided helper)."""
    nrows = int(np.ceil(len(idx) / ncols))
    plt.figure(figsize=(2 * ncols, 2.2 * nrows))
    for k, i in enumerate(idx):
        plt.subplot(nrows, ncols, k + 1)
        plt.imshow(X_raw[i].reshape(8, 8), cmap="gray_r")
        plt.title(titles[k], fontsize=9); plt.axis("off")
    plt.tight_layout(); plt.show()

print(X_dev.shape, X_test.shape, y_dev.mean())    # (1437, 64) (360, 64) about 0.50

<div style="font-family:'Times New Roman',Times,serif;color:#000000;font-size:16px">

## Task 1 &mdash; Build the flexible MLP

> **Concept.** Your Lab 6 network had fixed choices: sigmoid hidden units, BCE loss and full-batch
> updates. To study these as hyperparameters you must make them switchable.

**Task 1.1.** Print the fraction of odd digits (already above), and use `show_images` to display ten
digits from `X_dev` with their real digit as the title.

**Task 1.2.** Implement `forward`, `backward` and `train` below so that they support **all** options in the menu.
The helpers (`sigmoid`, `bce`, `mse`, `init_params`, `predict`, `accuracy`) are provided.

**Task 1.3.** Gradient check. For every new option that changes `backward` (tanh, relu, mse, l2),
compare your analytic gradients with `numerical_gradient` on a small subset (20 examples, hidden = 4).
Print PASS or FAIL for each of `W1, b1, W2, b2` (relative error below `1e-6`). (If ReLU fails on your machine, try another seed in `init_params` and explain why in Q1.)

**Task 1.4.** Run the **baseline** once on a stratified 75% / 25% split of `X_dev` (`random_state = SEED`).
Plot the training loss (and the validation loss if you track it) against the epoch, and print the training
and validation accuracy.

</div>

In [ ]:
# ---- Task 1.1: look at the data ----
# TODO 1.1 — show_images(X_dev, [str(d) for d in d_dev[:10]], range(10))

In [ ]:
# ---- Provided helpers ----
def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))

def bce(y_true, p_pred):
    p = np.clip(p_pred, 1e-12, 1 - 1e-12)
    return -np.mean(y_true * np.log(p) + (1 - y_true) * np.log(1 - p))

def mse(y_true, p_pred):
    return np.mean((y_true - p_pred) ** 2)

def data_loss(y_true, p_pred, loss="bce"):
    return bce(y_true, p_pred) if loss == "bce" else mse(y_true, p_pred)

def init_params(n_in, hidden, init_scale=0.5, seed=0):
    rng = np.random.default_rng(seed)
    W1 = rng.normal(0, 1, (hidden, n_in)) * init_scale
    b1 = np.zeros(hidden)
    W2 = rng.normal(0, 1, (1, hidden)) * init_scale
    b2 = np.zeros(1)
    return [W1, b1, W2, b2]

def numerical_gradient(param, loss_fn, eps=1e-5):
    grad = np.zeros_like(param)
    for idx in np.ndindex(param.shape):
        old = param[idx]
        param[idx] = old + eps;  loss_plus = loss_fn()
        param[idx] = old - eps;  loss_minus = loss_fn()
        param[idx] = old
        grad[idx] = (loss_plus - loss_minus) / (2 * eps)
    return grad

In [ ]:
def forward(X, params, activation="sigmoid"):
    # TODO 1.2 — Z1, A1 (sigmoid / tanh / relu), Z2, A2 = sigmoid(Z2)
    # return A2 and cache = (X, A1, A2)
    raise NotImplementedError

def backward(y, cache, params, activation="sigmoid", loss="bce", l2=0.0):
    X, A1, A2 = cache
    W1, b1, W2, b2 = params
    m = X.shape[0]
    # TODO 1.2 — dZ2 (bce or mse), dW2, db2, dA1, dZ1 (depends on activation), dW1, db1
    #            then add the L2 terms (lam/m)*W to dW1 and dW2
    # TODO 1.2 — assert each gradient has the same shape as its parameter
    raise NotImplementedError

def train(X, y, hidden=16, lr=0.1, epochs=500, activation="sigmoid", loss="bce",
          batch_size=None, l2=0.0, init_scale=0.5, seed=0, X_val=None, y_val=None):
    params = init_params(X.shape[1], hidden, init_scale, seed)
    rng = np.random.default_rng(seed + 1)          # separate generator for shuffling
    m = X.shape[0]
    train_losses, val_losses = [], []
    # TODO 1.2 — for each epoch:
    #   * if batch_size is None, use all m examples as one batch; otherwise shuffle the indices
    #     with rng.permutation(m) and loop over chunks of batch_size examples
    #   * for each batch: forward, backward, update all four parameters (param -= lr * grad)
    #   * at the END of the epoch: store the data loss on the whole training set in train_losses
    #     (and on X_val, y_val in val_losses if they are given)
    return params, train_losses, val_losses

def predict(X, params, activation="sigmoid"):
    return (forward(X, params, activation)[0] >= 0.5).astype(int)

def accuracy(X, y, params, activation="sigmoid"):
    return np.mean(predict(X, params, activation) == y)

In [ ]:
# ---- Task 1.3: gradient check on a small subset ----
X_check = standardise(X_dev[:20])[0]
y_check = y_dev[:20]

def check_option(activation, loss, l2, hidden=4):
    params = init_params(X_check.shape[1], hidden, 0.5, seed=3)

    def loss_fn():                                  # data loss + L2 penalty, reads the SAME arrays
        A2, _ = forward(X_check, params, activation)
        total = data_loss(y_check, A2, loss)
        total += (l2 / (2 * len(X_check))) * (np.sum(params[0] ** 2) + np.sum(params[2] ** 2))
        return total

    # TODO 1.3 — analytic gradients from forward + backward; numerical ones from
    # numerical_gradient(param, loss_fn) for each of W1, b1, W2, b2; then print the relative
    # error  norm(a-n) / (norm(a)+norm(n)+1e-12)  and PASS / FAIL for each (PASS if < 1e-6)
    raise NotImplementedError

for act, ls, lam in [("sigmoid", "bce", 0.0), ("tanh", "bce", 0.0), ("relu", "bce", 0.0),
                     ("sigmoid", "mse", 0.0), ("sigmoid", "bce", 1.0)]:
    print(f"--- activation={act}, loss={ls}, l2={lam} ---")
    check_option(act, ls, lam)

In [ ]:
# ---- Task 1.4: baseline on one 75% / 25% split ----
# TODO 1.4 — split X_dev, y_dev (stratified, test_size=0.25, random_state=SEED),
#            standardise (fit on the training part only), train with BASELINE (pass X_val, y_val),
#            plot the loss curve(s), print training and validation accuracy

# Task 1 answers:
# Q1 (why can ReLU fail a numerical gradient check at some points even when backward() is correct?
#     hint: Lab 3, Task 2):
# Q2 (why standardise the pixels before a sigmoid network? think about unscaled inputs, Z1 and
#     the sigmoid gradient, Lab 3 Task 3):

<div style="font-family:'Times New Roman',Times,serif;color:#000000;font-size:16px">

## Task 2 &mdash; Cross-validation and the baseline score

> **Concept.** From now on every number you report comes from 5-fold cross-validation on the
> development set, not from a single split.

**Task 2.1.** Write `cross_validate(config)`. For each of the 5 folds from `skf.split(X_dev, y_dev.ravel())`:
standardise using the **training fold only**, train with `config` (a dictionary of hyperparameters such as
`BASELINE`), and record the training accuracy, the validation accuracy and the validation **BCE** loss
(always BCE, so that settings are comparable). Return a dictionary with the keys `train_acc`, `val_acc`,
`val_loss` (each a tuple `(mean, std)`) and `oof_pred` (the predicted class of every development example
when it was in the validation fold, shape `(1437,)`).

**Task 2.2.** Run it on `BASELINE` and print the result as mean &plusmn; std. Compare with the self-check (about 89%).

**Task 2.3.** Using `oof_pred` and `d_dev`, find which real digits (0 to 9) are misclassified most often:
print the error rate for each digit.

</div>

In [ ]:
def cross_validate(config):
    # TODO 2.1 — loop over the 5 folds, standardise on the training fold only, train, evaluate
    # return {"train_acc": (mean, std), "val_acc": (mean, std), "val_loss": (mean, std), "oof_pred": ...}
    raise NotImplementedError

# TODO 2.2 — res_base = cross_validate(BASELINE); print train_acc, val_acc, val_loss as mean +- std
# TODO 2.3 — error rate per real digit, from res_base["oof_pred"], y_dev and d_dev

# Task 2 answers:
# Q1 (why must standardisation be fitted on the training fold only, not on all 1437 examples?):
# Q2 (the std of the baseline is about 2%: how should you read a difference of 0.5% between two settings?):

<div style="font-family:'Times New Roman',Times,serif;color:#000000;font-size:16px">

## Task 3 &mdash; Plan your ablation study

> **Concept.** Before running experiments, decide what you will test and what you expect. A hypothesis
> is a prediction **with a reason**. Example: *"If I lower the learning rate from 0.1 to 0.001, I expect lower
> accuracy after 500 epochs, because each step is 100 times smaller (Lab 5, Task 6)."* Writing predictions
> first lets you learn from the cases where you were wrong.

**Task 3.1.** Choose **at least six** hyperparameters from the menu, with **at least two from each of the groups A, B and C**.

**Task 3.2.** For each one, choose **at least four values** (including the baseline value) and write a one-sentence hypothesis with a reason.

**Task 3.3.** State the metric you will use to rank settings (CV validation accuracy, CV validation loss, or both) and why.

**Fill in this table** (double-click the cell to edit; add rows if you need them):

| Hyperparameter | Values to test | Hypothesis (what do you expect, and why?) |
|---|---|---|
|  |  |  |
|  |  |  |
|  |  |  |
|  |  |  |
|  |  |  |
|  |  |  |

**Metric I will use, and why:**

&nbsp;

</div>

In [ ]:
# Task 3 answers:
# Q1 (why change only one hyperparameter at a time? what could you not conclude if you changed lr and hidden together?):
# Q2 (which metric did you choose and why? can validation loss show a difference that accuracy hides?
#     think of two models that both get 90% right, one far more confident when it is wrong):

<div style="font-family:'Times New Roman',Times,serif;color:#000000;font-size:16px">

## Task 4 &mdash; Run the ablation study

> **Concept.** Now run your plan. For each hyperparameter, change **only that one**, cross-validate each
> value, and record the result.

**Task 4.1.** Fill in `ablation_plan` with the hyperparameters and values from your Task 3 table. Loop over
it: for each value run `cross_validate({**BASELINE, name: value})` and log the result with the provided `log_result`.

**Task 4.2.** Show **all** results in one table (`pd.DataFrame(results)`), including the baseline row.

**Task 4.3.** Draw **one plot per group** (A, B, C) with the provided `plot_ablation` (CV validation accuracy against
the value, with error bars). For the learning rate also draw the training loss curves of the different values on one axes
(use `train` directly on one split for this).

**Task 4.4.** Rank your hyperparameters from the most to the least influential, using the **range** of CV validation
accuracy (best value minus worst value) for each. Show the ranking as a small table.

</div>

In [ ]:
results = []
def log_result(hyper, value, res):
    """Add one row to the results table (provided)."""
    results.append({"hyper": hyper, "value": str(value),
                    "train_acc": round(res["train_acc"][0], 4),
                    "val_mean": round(res["val_acc"][0], 4), "val_std": round(res["val_acc"][1], 4),
                    "val_loss": round(res["val_loss"][0], 4),
                    "gap": round(res["train_acc"][0] - res["val_acc"][0], 4)})

def plot_ablation(df, hypers, title):
    """One panel per hyperparameter: CV val. accuracy vs value with +-std error bars (provided)."""
    fig, axes = plt.subplots(1, len(hypers), figsize=(5 * len(hypers), 3.8), squeeze=False)
    for ax, h in zip(axes[0], hypers):
        d = df[df["hyper"] == h]
        ax.errorbar(range(len(d)), d["val_mean"], yerr=d["val_std"], marker="o", capsize=4, label="CV val. accuracy")
        ax.set_xticks(range(len(d))); ax.set_xticklabels(d["value"])
        ax.set_xlabel(h); ax.set_ylabel("accuracy"); ax.set_title(h); ax.legend()
    fig.suptitle(title); plt.tight_layout(); plt.show()

# TODO 4.1 — ablation_plan = {"lr": [0.001, 0.01, 0.1, 1.0], ...}   (your six or more hyperparameters)
ablation_plan = {}

log_result("baseline", "-", res_base)                  # the baseline row
# TODO 4.1 — for name, values in ablation_plan.items(): for v in values: run cross_validate and log_result
#            (for batch_size = 1 use fewer epochs, for example 50, and note it in your report)

# TODO 4.2 — display(pd.DataFrame(results))
# TODO 4.3 — plot_ablation(df, [...], "Group A: optimisation"), then groups B and C;
#            plus the learning-rate loss curves on one axes
# TODO 4.4 — ranking table: for each hyperparameter, max(val_mean) - min(val_mean), sorted from largest to smallest

In [ ]:
# Task 4 answers:
# Q1 (which hyperparameter had the biggest effect, which almost none? explain with Labs 3-6 theory):
# Q2 (did any result contradict your hypothesis from Task 3? what did you expect, what happened, why?):
# Q3 (for which results is the difference smaller than the std? what can you say about those?):
# Q4 (was there a hyperparameter where training accuracy kept rising but validation accuracy did not? what is that called?):

<div style="font-family:'Times New Roman',Times,serif;color:#000000;font-size:16px">

## Task 5 &mdash; Combine the best settings

> **Concept.** Taking the best value of each hyperparameter separately does not always give the best model
> together, because hyperparameters **interact**. For example, a larger learning rate may work well only with
> fewer epochs.

**Task 5.1.** Build a combined configuration from the best value of each hyperparameter in Task 4. Cross-validate it
and compare it with the baseline.

**Task 5.2.** Try at most **five more** configurations of your own choice (for example, changing the two most influential
hyperparameters together). Record all of them in a table. **Do not exceed this budget.**

**Task 5.3.** Pick your **final configuration using only CV results**, and write two or three sentences explaining the choice.
If a simpler configuration is within one std of the best, say which you chose and why.

</div>

In [ ]:
# TODO 5.1 — combined = {**BASELINE, ...best values...}; res_comb = cross_validate(combined)
# TODO 5.2 — at most five more configurations; record them in a table (DataFrame)
# TODO 5.3 — FINAL = {...}   (chosen from CV results only)

# Task 5 answers:
# Q1 (was the combined configuration better than the baseline and than each single change? if not, what
#     interaction between hyperparameters could explain it?):
# My final configuration and why (2-3 sentences):

<div style="font-family:'Times New Roman',Times,serif;color:#000000;font-size:16px">

## Task 6 &mdash; Final test

> **Concept.** This is the only task where the test set is used, and it is used **once**. If the result disappoints
> you, do **not** go back and change the configuration; report it honestly.

**Task 6.1.** Retrain the **baseline** and your **`FINAL`** configuration on all 1437 development examples
(standardise using the development set only).

**Task 6.2.** Evaluate both on the 360 test examples. Report the accuracy, the BCE loss and the number of wrong
predictions in a table. Then show **five** test images that your final model got wrong, with the true digit and the
predicted class (odd or even) in each title (use `show_images`).

**Task 6.3.** Compare the test accuracy of the final model with its CV accuracy.

</div>

In [ ]:
# TODO 6.1 — standardise (fit on X_dev), train baseline and FINAL on all of X_dev
# TODO 6.2 — test accuracy, test BCE loss, number of wrong predictions for both; table
#            show_images(X_test, [...titles...], wrong_indices[:5])
# TODO 6.3 — compare with the CV accuracy of FINAL

# Task 6 answers:
# Q1 (is the test accuracy close to the CV estimate? if not, why? the test set has 360 examples:
#     how much does one wrong prediction change the accuracy?):
# Q2 (look at the five wrong images: would a person also find them hard? which real digits does the model
#     confuse most, and why might an 8x8 image make them hard to tell apart?):

<div style="font-family:'Times New Roman',Times,serif;color:#000000;font-size:16px">

## Task 7 &mdash; Justification summary

Write a short summary (half to one page) **in your own words**, as if you were reporting to a project lead who wants to
use your model. Double-click this cell and write below the line. Your summary must answer:

1. Which hyperparameters mattered most and which hardly mattered? Give the key numbers.
2. Why do you think the most important one mattered so much on this dataset? Link it to a lab topic.
3. Which of your hypotheses were wrong, and what did you learn from that?
4. Would you trust this model to read postal codes in a real post office? Give two limitations
   (for example the tiny 8 &times; 8 images, the number of configurations tried, the choice of seed, or writing styles not seen in the data).
5. If you had more time, what would you test next?

---

**My summary:**

&nbsp;

</div>

<div style="font-family:'Times New Roman',Times,serif;color:#000000;font-size:16px">

---

### Before you submit

- [ ] Kernel &rarr; Restart &amp; Run All, no errors.
- [ ] Every `TODO` and every `raise NotImplementedError` gone.
- [ ] Task 1: all new options print PASS in the gradient check; baseline loss curve and accuracies shown.
- [ ] Task 2: baseline CV is close to 89%; per-digit error rates printed.
- [ ] Task 3: plan table complete (six hyperparameters, four values each, hypotheses) and metric explained.
- [ ] Task 4: complete ablation table, one plot per group, ranking table.
- [ ] Task 5: at most six combined / extra configurations; final configuration chosen from CV only.
- [ ] Task 6: test set used once; table and five wrong images shown.
- [ ] Task 7: summary written. Written answers present as comments: 1 (Q1&ndash;Q2), 2 (Q1&ndash;Q2), 3 (Q1&ndash;Q2), 4 (Q1&ndash;Q4), 5 (Q1), 6 (Q1&ndash;Q2).
- [ ] Renamed to `Lab7_<yourCMSID>.ipynb`.

### One thing to take away

Every number you used in earlier labs (`hidden`, `lr`, `epochs`, ...) was a choice. Now you know how to
test such choices fairly: one change at a time, cross-validated, with the test set kept for the very end.

</div>